# 15 - AWS Few-Shot Calibration (three-stage validation of the proposed pipeline)

**Jalankan di SageMaker SETELAH `cleaned_100.pkl` + UNSW CSV + AWS labeled CSV tersedia.**

Menjawab review #6: bukti bahwa pipeline USULAN (source + 1% target + adversarial training)
benar-benar transfer ke trafik cloud AWS nyata, bukan sekadar zero-shot domain-shift failure.

Tiga tahap per arah (CIC->AWS dan UNSW->AWS):
- **S0 zero-shot**: model source -> eval AWS (reproduksi kegagalan domain-shift).
- **S1 +1% AWS calib**: source + 1% label AWS (few-shot) -> eval AWS clean.
- **S2 +1% AWS + adv**: S1 + adversarial training -> eval AWS clean & adversarial (PCFS FGSM/PGD).

Leakage control: few-shot diambil HANYA dari partisi train AWS; evaluasi pada partisi test AWS yang disjoint.
Rerata 5 seed. Angka NYATA -> ditulis ke `paper2_reviewer_out/` + baris LaTeX untuk `tab:aws_stages`.

In [ ]:
import importlib, subprocess, sys
for pkg,imp in [('pandas','pandas'),('numpy','numpy'),('scikit-learn','sklearn'),('xgboost','xgboost'),('scipy','scipy'),('boto3','boto3')]:
    try: importlib.import_module(imp)
    except ImportError: subprocess.check_call([sys.executable,'-m','pip','install','-q',pkg])
import os, json, pickle
import numpy as np, pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import matthews_corrcoef, f1_score, precision_score, recall_score, confusion_matrix
from xgboost import XGBClassifier
from scipy import stats

CIC_PKL='../../CICDDoS2018/data/cleaned_100.pkl'
UNSW_TRAIN='../data/UNSW_NB15_testing-set.csv'; UNSW_TEST='../data/UNSW_NB15_training-set.csv'
AWS_CLEAN='aws_labeled/detect_clean_flows.csv'
AWS_VOL='aws_labeled/detect_volumetric_flows.csv'
OUTDIR='paper2_reviewer_out'; os.makedirs(OUTDIR,exist_ok=True)
S3_BUCKET=os.environ.get('S3_BUCKET','ssh-detection-features-232032302717'); REGION='ap-southeast-1'
SEEDS=[13,42,101,202,303]
H=0.01; EPS_TRAIN=0.1; ADV_RATIO=0.20; FEWSHOT_FRAC=0.01; MAXN=40000
EPS_EVAL=[0.05,0.1,0.2]; PGD_ITERS=10; PGD_ALPHA=0.02
print('files:', os.path.exists(CIC_PKL), os.path.exists(UNSW_TRAIN), os.path.exists(UNSW_TEST), os.path.exists(AWS_CLEAN), os.path.exists(AWS_VOL))

## 1. SFM mapping + util (identik nb14 supaya konsisten)

In [ ]:
MAP_A={'duration':('Flow Duration','dur'),'fwd_pkts':('Tot Fwd Pkts','spkts'),
       'bwd_pkts':('Tot Bwd Pkts','dpkts'),'fwd_bytes':('TotLen Fwd Pkts','sbytes'),
       'bwd_bytes':('TotLen Bwd Pkts','dbytes'),'fwd_mean':('Fwd Pkt Len Mean','smean'),
       'bwd_mean':('Bwd Pkt Len Mean','dmean'),'src_load':('Flow Byts/s','sload'),
       'dst_load':('Bwd Pkts/s','dload')}
CANON=list(MAP_A.keys()); IX={c:i for i,c in enumerate(CANON)}
def build_matrix(df,side):
    idx=0 if side=='cic' else 1; cols=[MAP_A[c][idx] for c in CANON]
    out=df[cols].copy(); out.columns=CANON; out=out.replace([np.inf,-np.inf],np.nan)
    return out.fillna(out.median(numeric_only=True)).fillna(0.0).astype(float).values
def make_xgb(seed):
    return XGBClassifier(objective='binary:logistic',eval_metric='logloss',max_depth=8,
        learning_rate=0.1,n_estimators=200,subsample=0.8,colsample_bytree=0.8,
        n_jobs=-1,random_state=seed,tree_method='hist')
def loss_bin(model,X,y):
    p=np.clip(model.predict_proba(X)[:,1],1e-15,1-1e-15); y=y.astype(float)
    return -(y*np.log(p)+(1-y)*np.log(1-p))
def saliency(model,X,y,h=H):
    n,m=X.shape; S=np.zeros((n,m))
    for i in range(m):
        Xp=X.copy(); Xp[:,i]+=h; Xm=X.copy(); Xm[:,i]-=h
        S[:,i]=(loss_bin(model,Xp,y)-loss_bin(model,Xm,y))/(2*h)
    return S
def project_functional(Xs_scaled, mean, scale, X_ref_scaled=None):
    Xo=Xs_scaled*scale+mean; Xo=np.clip(Xo,0.0,None)
    Xo[:,IX['fwd_pkts']]=np.round(Xo[:,IX['fwd_pkts']]); Xo[:,IX['bwd_pkts']]=np.round(Xo[:,IX['bwd_pkts']])
    Xo[:,IX['fwd_bytes']]=np.maximum(Xo[:,IX['fwd_bytes']],Xo[:,IX['fwd_pkts']])
    Xo[:,IX['bwd_bytes']]=np.maximum(Xo[:,IX['bwd_bytes']],Xo[:,IX['bwd_pkts']])
    if X_ref_scaled is not None:
        Xr=X_ref_scaled*scale+mean
        for j in [IX[c] for c in ['fwd_pkts','bwd_pkts','fwd_bytes','bwd_bytes','duration']]:
            Xo[:,j]=np.maximum(Xo[:,j],Xr[:,j])
    with np.errstate(divide='ignore',invalid='ignore'):
        fm=np.where(Xo[:,IX['fwd_pkts']]>0,Xo[:,IX['fwd_bytes']]/Xo[:,IX['fwd_pkts']],0.0)
        bm=np.where(Xo[:,IX['bwd_pkts']]>0,Xo[:,IX['bwd_bytes']]/Xo[:,IX['bwd_pkts']],0.0)
    Xo[:,IX['fwd_mean']]=fm; Xo[:,IX['bwd_mean']]=bm
    with np.errstate(divide='ignore',invalid='ignore'):
        dd=np.where(Xo[:,IX['duration']]>0,Xo[:,IX['duration']],np.nan)
        Xo[:,IX['src_load']]=np.nan_to_num((Xo[:,IX['fwd_bytes']]+Xo[:,IX['bwd_bytes']])/dd,nan=0.0)
        Xo[:,IX['dst_load']]=np.nan_to_num(Xo[:,IX['bwd_pkts']]/dd,nan=0.0)
    return (Xo-mean)/scale
def fgsm_functional(X,S,eps,mean,scale):
    return project_functional(X+eps*np.sign(S),mean,scale,X_ref_scaled=X)
def pgd_functional(model,X,y,eps,mean,scale,iters=PGD_ITERS,alpha=PGD_ALPHA):
    Xadv=X.copy()
    for _ in range(iters):
        S=saliency(model,Xadv,y)
        Xadv=Xadv+alpha*np.sign(S)
        Xadv=np.clip(Xadv,X-eps,X+eps)
        Xadv=project_functional(Xadv,mean,scale,X_ref_scaled=X)
    return Xadv
def cm_metrics(y,yp):
    tn,fp,fn,tp=confusion_matrix(y,yp,labels=[0,1]).ravel()
    return dict(mcc=float(matthews_corrcoef(y,yp)),f1=float(f1_score(y,yp,zero_division=0)),
                recall=float(recall_score(y,yp,zero_division=0)),
                precision=float(precision_score(y,yp,zero_division=0)),
                tp=int(tp),fp=int(fp),fn=int(fn),tn=int(tn))
def asr(y,yp_clean,yp_adv):
    atk=(y==1); det_before=atk & (yp_clean==1)
    if det_before.sum()==0: return float('nan')
    evaded=det_before & (yp_adv==0)
    return float(evaded.sum()/det_before.sum())
print('util siap')

## 2. Muat data source (benchmark) + AWS labeled

AWS labeled: gabung `detect_clean_flows.csv` + `detect_volumetric_flows.csv`; sudah 9 fitur SFM + ground_truth.

In [ ]:
with open(CIC_PKL,'rb') as f: d=pickle.load(f)
cic_feats=list(d['feature_names']); X=np.asarray(d['X'],float); sc=d.get('scaler',None)
X_orig=X*sc.scale_+sc.mean_ if (sc is not None and hasattr(sc,'scale_')) else X
cic_df=pd.DataFrame(X_orig,columns=cic_feats)
benign=d.get('label_mapping',{}).get('Benign',0); y_cic=(np.asarray(d['y'])!=benign).astype(int)
unsw_tr=pd.read_csv(UNSW_TRAIN); y_utr=unsw_tr['label'].astype(int).values
Xc_all=build_matrix(cic_df,'cic'); y_cic=y_cic.astype(int)
Xu_all=build_matrix(unsw_tr,'unsw')
# AWS labeled (fitur sudah kanonik) -> matriks 9 fitur + label
aws=pd.concat([pd.read_csv(AWS_CLEAN),pd.read_csv(AWS_VOL)],ignore_index=True)
aws=aws.replace([np.inf,-np.inf],np.nan)
Xa_raw=aws[CANON].fillna(aws[CANON].median(numeric_only=True)).fillna(0.0).astype(float).values
ya=aws['ground_truth'].astype(int).values
print('CIC',Xc_all.shape,'UNSW',Xu_all.shape,'AWS',Xa_raw.shape,'AWS attack:benign',int(ya.sum()),int((ya==0).sum()))

## 3. Tiga tahap x dua arah x 5 seed

Per arah: source = CIC (atau UNSW). Scaler di-fit pada source-train saja; AWS di-transform.
AWS dipartisi train/test stratified per seed; few-shot 1% diambil dari AWS-train saja (anti-leakage).

In [ ]:
def robust_augment(base_model,Xtr,ytr,rng):
    n=min(MAXN,len(Xtr)); idx=rng.choice(len(Xtr),n,replace=False)
    S=saliency(base_model,Xtr[idx],ytr[idx]); Xa=Xtr[idx]+EPS_TRAIN*np.sign(S)
    na=min(int(len(Xtr)*ADV_RATIO/(1-ADV_RATIO)),len(Xa)); sel=rng.choice(len(Xa),na,replace=False)
    return np.vstack([Xtr,Xa[sel]]),np.concatenate([ytr,ytr[idx][sel]])

def eval_adv_block(stage,direction,seed,model,mean,scale,Xte,yte,rows):
    yp_clean=model.predict(Xte)
    r={'direction':direction,'seed':seed,'stage':stage,'condition':'clean'}; r.update(cm_metrics(yte,yp_clean)); r['asr']=float('nan'); rows.append(r)
    if stage=='S2_fewshot_adv':
        S=saliency(model,Xte,yte)
        for e in EPS_EVAL:
            Xf=fgsm_functional(Xte,S,e,mean,scale); ypf=model.predict(Xf)
            rf={'direction':direction,'seed':seed,'stage':stage,'condition':f'fgsm_eps{e}'}; rf.update(cm_metrics(yte,ypf)); rf['asr']=asr(yte,yp_clean,ypf); rows.append(rf)
            Xp=pgd_functional(model,Xte,yte,e,mean,scale); ypp=model.predict(Xp)
            rp={'direction':direction,'seed':seed,'stage':stage,'condition':f'pgd_eps{e}'}; rp.update(cm_metrics(yte,ypp)); rp['asr']=asr(yte,yp_clean,ypp); rows.append(rp)
    return yp_clean

def run_direction(direction,Xsrc_raw,ysrc):
    rows=[]
    for seed in SEEDS:
        rng=np.random.RandomState(seed)
        # scaler fit pada source saja (anti-leakage), AWS di-transform
        scaler=StandardScaler().fit(Xsrc_raw); Xsrc=scaler.transform(Xsrc_raw)
        mean,scale=scaler.mean_,scaler.scale_
        # partisi AWS train/test stratified (disjoint); few-shot dari AWS-train saja
        Xa_tr_raw,Xa_te_raw,ya_tr,ya_te=train_test_split(Xa_raw,ya,test_size=0.5,random_state=seed,stratify=ya)
        Xa_tr=scaler.transform(Xa_tr_raw); Xa_te=scaler.transform(Xa_te_raw)
        # ---- S0 zero-shot: latih source saja, eval AWS-test
        m0=make_xgb(seed).fit(Xsrc,ysrc)
        eval_adv_block('S0_zeroshot',direction,seed,m0,mean,scale,Xa_te,ya_te,rows)
        # ---- S1 few-shot: source + 1% AWS-train berlabel
        nfs=max(1,int(len(Xa_tr)*FEWSHOT_FRAC)); ifs=rng.choice(len(Xa_tr),nfs,replace=False)
        Xfs=np.vstack([Xsrc,Xa_tr[ifs]]); yfs=np.concatenate([ysrc,ya_tr[ifs]])
        m1=make_xgb(seed).fit(Xfs,yfs)
        eval_adv_block('S1_fewshot',direction,seed,m1,mean,scale,Xa_te,ya_te,rows)
        # ---- S2 few-shot + adversarial training
        Xr,yr=robust_augment(m1,Xfs,yfs,rng); m2=make_xgb(seed).fit(Xr,yr)
        eval_adv_block('S2_fewshot_adv',direction,seed,m2,mean,scale,Xa_te,ya_te,rows)
        print('  ',direction,'seed',seed,'nfs',nfs,'AWS te',len(ya_te),'selesai')
    return rows

ALL=[]
ALL+=run_direction('CIC->AWS',Xc_all,y_cic)
ALL+=run_direction('UNSW->AWS',Xu_all,y_utr)
dfa=pd.DataFrame(ALL); dfa.to_csv(os.path.join(OUTDIR,'aws_stages_raw.csv'),index=False)
print('total rows',len(dfa))

## 4. Agregasi (mean/std/95% CI) + tabel LaTeX untuk paper (tab:aws_stages)

In [ ]:
def ci95(x):
    x=np.asarray(x,float); x=x[np.isfinite(x)]
    if len(x)<2: return (float('nan'),float('nan'))
    m=x.mean(); h=stats.t.ppf(0.975,len(x)-1)*x.std(ddof=1)/np.sqrt(len(x)); return (m-h,m+h)
agg=dfa.groupby(['direction','stage','condition']).agg(
    mcc_mean=('mcc','mean'),mcc_std=('mcc','std'),asr_mean=('asr','mean'),
    recall_mean=('recall','mean'),precision_mean=('precision','mean'),f1_mean=('f1','mean')).reset_index()
ci=dfa.groupby(['direction','stage','condition'])['mcc'].apply(lambda s: ci95(s.values)).reset_index(name='mcc_ci95')
agg=agg.merge(ci,on=['direction','stage','condition'])
agg.to_csv(os.path.join(OUTDIR,'aws_stages_agg.csv'),index=False)
import IPython.display as ipd; ipd.display(agg.round(4))
json.dump(agg.round(4).to_dict(orient='records'),open(os.path.join(OUTDIR,'aws_stages_agg.json'),'w'),indent=2)
print('=== agregasi AWS stages selesai ===')

In [ ]:
# --- Baris LaTeX untuk tab:aws_stages (clean MCC + recall/precision per tahap; kolom adversarial utk S2) ---
STAGES=[('S0_zeroshot','S0: source$\\rightarrow$AWS (zero-shot)'),
        ('S1_fewshot','S1: $+1\\%$ AWS few-shot'),
        ('S2_fewshot_adv','S2: $+1\\%$ AWS $+$ adv')]
DIRS=['CIC->AWS','UNSW->AWS']
def g(direction,stage,cond,col):
    s=agg[(agg['direction']==direction)&(agg['stage']==stage)&(agg['condition']==cond)]
    return float(s[col].iloc[0]) if len(s) else float('nan')
def fmt(v):
    if not np.isfinite(v): return '--'
    return '$'+f'{v:.3f}'.replace('.',',')+'$'
lines=[]
for direction in DIRS:
    lines.append('\\multirow{3}{*}{'+direction.replace('->','$\\rightarrow$')+'}')
    for i,(sk,sl) in enumerate(STAGES):
        clean_mcc=fmt(g(direction,sk,'clean','mcc_mean'))
        rec=fmt(g(direction,sk,'clean','recall_mean')); prec=fmt(g(direction,sk,'clean','precision_mean'))
        advc=fmt(g(direction,sk,'fgsm_eps0.1','mcc_mean')) if sk=='S2_fewshot_adv' else '--'
        advp=fmt(g(direction,sk,'pgd_eps0.1','mcc_mean')) if sk=='S2_fewshot_adv' else '--'
        lines.append(f' & {sl} & {clean_mcc} & {rec} & {prec} & {advc} & {advp} \\\\')
    lines.append('\\midrule' if direction!=DIRS[-1] else '')
latex='\n'.join(l for l in lines if l!='')
open(os.path.join(OUTDIR,'aws_stages_rows.tex'),'w').write(latex)
print('=== TEMPEL baris berikut ke Tabel tab:aws_stages ===\n'); print(latex)
print('\n=== file: aws_stages_agg.csv + aws_stages_rows.tex di',OUTDIR,'===')

In [ ]:
# Upload artefak ke S3
try:
    import boto3; s3=boto3.client('s3',region_name=REGION); up=0
    for fn in sorted(os.listdir(OUTDIR)):
        if fn.startswith('aws_stages') and fn.endswith(('.csv','.json','.tex')):
            s3.upload_file(os.path.join(OUTDIR,fn),S3_BUCKET,f'unsw-far/paper2_reviewer/{fn}'); up+=1
    print('upload',up,'-> s3://%s/unsw-far/paper2_reviewer/'%S3_BUCKET)
except Exception as e: print('upload gagal:',e)